# Implementasi Fine-Tuning Qwen 2.5 (3B) dengan QLoRA (4-bit NF4) & Hugging Face TRL

Notebook ini berisi implementasi lengkap fine-tuning LLM (Large Language Model) **Qwen 2.5-3B-Instruct** untuk tugas **Credit Risk Underwriting & Explainable AI (XAI)**.

### Alur Pipeline:
1. **Pengecekan GPU & Manajemen VRAM**: Monitoring spesifikasi hardware dan alokasi memori CUDA.
2. **Kuantisasi 4-bit NF4**: Memuat model dengan `bitsandbytes` untuk efisiensi VRAM (kompatibel GPU 4GB VRAM seperti GTX 1650).
3. **PEFT (QLoRA)**: Menyiapkan LoRA adapter rank-8 pada seluruh modul linear attention & feed-forward.
4. **Data Preparation**: Memuat dataset JSONL dan menyusun prompt berformat ChatML (`<|im_start|>` - `<|im_end|>`).
5. **Supervised Fine-Tuning (SFT)**: Melatih model menggunakan `TRL SFTConfig` & `SFTTrainer` dengan optimasi memory-efficient (`paged_adamw_8bit`, gradient checkpointing).
6. **Inference & JSON Validation**: Menguji kemampuan model dalam menghasilkan Memo Underwriting Kredit berformat JSON valid.
7. **Simpan Adapter & Merge Model**: Menyimpan bobot LoRA dan melakukan merge ke FP16 untuk inferensi produksi atau konversi GGUF.

In [1]:
# ==============================================================================
# 1. PENGECEKAN GPU & MANAJEMEN VRAM
# ==============================================================================
# Kode ini memeriksa ketersediaan CUDA GPU, menghitung kapasitas VRAM maksimum,
# dan mencatat pemakaian awal VRAM sebelum model di-load ke memori.

import os
import sys
import torch

print(f"Versi Python : {sys.version.split()[0]}")
print(f"Versi PyTorch: {torch.__version__}")

# Cek ketersediaan GPU CUDA
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    total_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
    reserved_vram_gb = round(torch.cuda.memory_reserved(0) / (1024**3), 2)
    allocated_vram_gb = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
    
    print(f"[OK] GPU Terdeteksi : {device_name}")
    print(f"[OK] Total VRAM     : {total_vram_gb} GB")
    print(f"[OK] VRAM Reserved  : {reserved_vram_gb} GB")
    print(f"[OK] VRAM Allocated : {allocated_vram_gb} GB")
else:
    print("[WARNING] CUDA GPU tidak terdeteksi! Training akan berjalan di CPU (sangat lambat).")


Versi Python : 3.9.6
Versi PyTorch: 2.5.1+cu121
[OK] GPU Terdeteksi : NVIDIA GeForce GTX 1650
[OK] Total VRAM     : 4.0 GB
[OK] VRAM Reserved  : 0.0 GB
[OK] VRAM Allocated : 0.0 GB


### 2. Memuat Base Model dan Tokenizer (4-bit NF4 Quantization)
Pada tahap ini, kita memuat model **Qwen/Qwen2.5-3B-Instruct** menggunakan konfigurasi **4-bit NormalFloat (NF4)** melalui `bitsandbytes`:
- **4-bit NF4**: Mengompres bobot model 3B dari ~6 GB (FP16) menjadi hanya ~2.2 GB VRAM sehingga muat dan stabil di GPU 4GB VRAM.
- **Double Quantization**: Mengompresi konstanta kuantisasi untuk menghemat ~0.4 bit per parameter.
- **Device Map `{"": 0}`**: Memaksa seluruh modul model dimuat ke GPU 0 untuk menghindari pemecahan (*dispatch*) layer ke CPU oleh `accelerate`.

In [2]:
# ==============================================================================
# 2. LOAD BASE MODEL & TOKENIZER DENGAN 4-BIT NF4
# ==============================================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# Bersihkan cache VRAM sebelum memuat model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_id = "Qwen/Qwen2.5-3B-Instruct"

# 2.1 Konfigurasi Kuantisasi 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Aktifkan kuantisasi 4-bit
    bnb_4bit_quant_type="nf4",             # Menggunakan NormalFloat4 (optimal untuk LLM)
    bnb_4bit_compute_dtype=torch.float16,  # Tipe komputasi matriks saat forward/backward
    bnb_4bit_use_double_quant=True,        # Double quantization untuk efisiensi ekstra VRAM
    llm_int8_enable_fp32_cpu_offload=True, # Fallback offload jika VRAM terbatas
)

# 2.2 Load Tokenizer
print(f"Memuat tokenizer untuk model '{model_id}'...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    padding_side="right",                  # Wajib 'right' untuk causal LM training
)

# Pastikan token PAD terdefinisi (jika None, gunakan EOS token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2.3 Tentukan Device Map
# Menggunakan {"": 0} untuk memaksa seluruh model masuk ke GPU 0 (mencegah error offload CPU pada GPU 4GB)
device_map = {"": 0} if torch.cuda.is_available() else None

# 2.4 Load Base Model dengan Kuantisasi 4-bit
print(f"Memuat model base '{model_id}' dalam 4-bit NF4 ke GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True,
)

# 2.5 Persiapkan model untuk k-bit training
# - Membekukan (freeze) base weights
# - Mengubah layer norm ke FP32 demi stabilitas numerik
# - Mengaktifkan gradient checkpointing untuk hemat VRAM
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Nonaktifkan use_cache saat training (wajib jika gradient checkpointing aktif)
model.config.use_cache = False

print("\n[SUCCESS] Base Model & Tokenizer berhasil dimuat dalam 4-bit NF4!")


Memuat tokenizer untuk model 'Qwen/Qwen2.5-3B-Instruct'...
Memuat model base 'Qwen/Qwen2.5-3B-Instruct' dalam 4-bit NF4 ke GPU...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


[SUCCESS] Base Model & Tokenizer berhasil dimuat dalam 4-bit NF4!


### 3. Konfigurasi LoRA Adapter (PEFT)
Kita menyisipkan adapter **LoRA (Low-Rank Adaptation)** pada layer-layer Linear (Attention Projection & MLP Layers):
- Dengan LoRA, hanya `< 1%` parameter yang dilatih (`trainable params`), sementara `99%+` parameter dasar tetap dibekukan (`frozen`).
- Modul target mencakup seluruh proyeksi: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`.

In [3]:
# ==============================================================================
# 3. KONFIGURASI LoRA (Low-Rank Adaptation)
# ==============================================================================

from peft import LoraConfig, TaskType, get_peft_model

# 3.1 Definisikan hyperparameter LoRA
peft_config = LoraConfig(
    r=8,                                   # Rank LoRA (dimensi bottleneck matriks adapter)
    lora_alpha=64,                         # Skala pengali update matriks (biasanya 2 * r)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # Self-attention projection
        "gate_proj", "up_proj", "down_proj",       # MLP feed-forward layers
    ],
    lora_dropout=0.05,                     # Dropout rate untuk regularisasi adapter
    bias="none",                           # Tidak melatih bias parameter
    task_type=TaskType.CAUSAL_LM,          # Task tipe Causal Language Modeling
)

# 3.2 Pasang LoRA adapter pada model
model = get_peft_model(model, peft_config)

# 3.3 Tampilkan statistik parameter yang dapat dilatih
print("Statistik Parameter Trainable:")
model.print_trainable_parameters()


Statistik Parameter Trainable:
trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


### 4. Memuat Dataset & Formatting ChatML Prompt
Model **Qwen 2.5 Instruct** menggunakan format prompt standar **ChatML** (`<|im_start|>` dan `<|im_end|>`):
```text
<|im_start|>system
{system_instruction}<|im_end|>
<|im_start|>user
{applicant_data_and_shap_metrics}<|im_end|>
<|im_start|>assistant
{underwriting_memo_json}<|im_end|>
```
Pada sel ini, kita memuat dataset `credit_finetune_dataset.jsonl` dan melakukan pemetaan format teks prompt secara efisien.

In [4]:
# ==============================================================================
# 4. LOAD DATASET & FORMATTING CHATML PROMPT
# ==============================================================================

import os
from datasets import load_dataset

# 4.1 Pencarian path dataset secara dinamis (fleksibel dari root folder atau folder notebooks)
candidate_paths = [
    "data/credit_finetune_dataset.jsonl",
    "../data/credit_finetune_dataset.jsonl",
    os.path.join(os.getcwd(), "data", "credit_finetune_dataset.jsonl"),
    os.path.join(os.getcwd(), "..", "data", "credit_finetune_dataset.jsonl"),
]

dataset_path = None
for p in candidate_paths:
    if os.path.exists(p):
        dataset_path = os.path.abspath(p)
        break

if dataset_path is None:
    raise FileNotFoundError(
        f"File dataset 'credit_finetune_dataset.jsonl' tidak ditemukan! Pastikan file berada di folder data/"
    )

print(f"[OK] Memuat dataset dari: {dataset_path}")
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"[OK] Total data training: {len(dataset)} sampel")

# 4.2 Template Format Prompt ChatML Qwen
prompt_template = """<|im_start|>system
{}<|im_end|>
<|im_start|>user
{}<|im_end|>
<|im_start|>assistant
{}<|im_end|>"""

EOS_TOKEN = tokenizer.eos_token if tokenizer.eos_token else "<|im_end|>"

# 4.3 Fungsi mapping formatting
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Gabungkan instruction, input, dan output ke dalam format ChatML
        text = prompt_template.format(instruction, input_text, output)
        texts.append(text)
    return {"text": texts}

# 4.4 Terapkan formatting ke dataset
formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

# Tampilkan contoh 1 data yang telah diformat
print("\n=== CONTOH FORMATTED PROMPT (Sampel 1) ===")
print(formatted_dataset[0]["text"][:600] + "\n...")


[OK] Memuat dataset dari: d:\project\credit-risk\data\credit_finetune_dataset.jsonl
[OK] Total data training: 1200 sampel

=== CONTOH FORMATTED PROMPT (Sampel 1) ===
<|im_start|>system
Anda adalah Senior Credit Risk Underwriter AI di institusi perbankan. Tugas Anda adalah mengevaluasi aplikasi kredit pemohon berdasarkan data demografi, keuangan, hasil prediksi model XGBoost (Probability of Default), dan kontribusi faktor risiko matematis (SHAP Values).

Hasilkan laporan analisis kredit (Credit Underwriting Memo) yang terstruktur strictly dalam format JSON valid.<|im_end|>
<|im_start|>user
### PROFIL PEMOHON PINJAMAN:
- Usia Pemohon: 22 tahun
- Pendapatan Tahunan: $55,000
- Status Kepemilikan Rumah: RENT
- Lama Bekerja: 3.0 tahun
- Tujuan Pinjaman: PERSONAL
...


### 5. Setup Hyperparameter & Eksekusi Training (TRL SFTConfig & SFTTrainer)
Menggunakan **TRL (`Transformer Reinforcement Learning`) `SFTConfig`** dan **`SFTTrainer`** untuk menjalankan supervised fine-tuning.
Hyperparameter disesuaikan agar optimal dan aman dari risiko Out of Memory (OOM) pada GPU dengan VRAM 4GB-8GB:
- **`per_device_train_batch_size=1`** & **`gradient_accumulation_steps=8`**: Memberikan ukuran batch efektif = 8 tanpa membebani VRAM.
- **`optim="paged_adamw_8bit"`**: Optimizer 8-bit dengan mekanisme paging ke CPU RAM jika terjadi lonjakan memori.
- **`learning_rate=2e-4`** dengan **Cosine LR Scheduler** & warmup.
- **`max_length=1024`**: Panjang konteks ideal untuk prompt dan respon JSON memo underwriting.

In [ ]:
# ==============================================================================
# 5. SETUP SFTTRAINER & TRAINING EKSEKUSI
# ==============================================================================

import os
import torch
from trl import SFTConfig, SFTTrainer

# 5.1 Tentukan direktori output checkpoint
output_dir = os.path.abspath(os.path.join("..", "outputs", "credit_qwen_qlora")) if os.path.exists("../data") else os.path.abspath("outputs/credit_qwen_qlora")
os.makedirs(output_dir, exist_ok=True)

# 5.2 Konfigurasi Argumen Pelatihan SFT (TRL SFTConfig)
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,       # Batch size per GPU (1 untuk hemat VRAM pada GPU 4GB)
    gradient_accumulation_steps=8,       # Akumulasi gradien (efektif batch size = 1 * 8 = 8)
    warmup_steps=10,                     # Langkah pemanasan learning rate
    num_train_epochs=2,                  # Jumlah epoch pelatihan
    learning_rate=2e-4,                  # Nilai Learning Rate standar untuk QLoRA
    fp16=torch.cuda.is_available(),      # Menggunakan FP16 mixed precision jika GPU tersedia
    logging_steps=10,                    # Interval pencatatan log loss
    optim="paged_adamw_8bit",            # Optimizer 8-bit hemat VRAM
    weight_decay=0.01,                   # Regularisasi weight decay
    lr_scheduler_type="cosine",          # Penjadwal learning rate Cosine
    seed=42,                             # Reproducibility seed
    save_strategy="epoch",               # Simpan checkpoint per akhir epoch
    save_total_limit=2,                  # Batasi checkpoint tersimpan untuk hemat disk
    report_to="none",                    # Nonaktifkan reporting pihak ketiga (wandb/tensorboard)
    dataset_text_field="text",           # Field teks pada dataset yang dilatih
    max_length=1024,                     # Panjang sequence token maksimum
)

# 5.3 Inisialisasi SFTTrainer (menggunakan processing_class untuk tokenizer di TRL modern)
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=training_args,
)

# 5.4 Jalankan Proses Fine-Tuning
print("Memulai proses fine-tuning QLoRA...")
trainer_stats = trainer.train()

# 5.5 Tampilkan Ringkasan Hasil Training
print("\n" + "=" * 50)
print(f"[SUCCESS] Training Selesai!")
print(f"Final Training Loss: {trainer_stats.training_loss:.4f}")
print("=" * 50)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_12812\980858386.py:7: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  from trl import SFTConfig, SFTTrainer
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Memulai proses fine-tuning QLoRA...


### 6. Uji Coba Inference & Validasi Output JSON
Setelah model selesai dilatih, kita menguji kemampuannya untuk menginterpretasikan data pemohon pinjaman, prediksi Machine Learning (Probability of Default), dan nilai SHAP secara terstruktur ke dalam format JSON valid (**Credit Underwriting Memo**).

In [ ]:
# ==============================================================================
# 6. INFERENCE TESTING & JSON SCHEMA VALIDATION
# ==============================================================================

import json
import torch

# 6.1 Set model ke mode evaluasi & aktifkan cache untuk inferensi cepat
model.eval()
model.config.use_cache = True

# 6.2 Contoh Instruksi System & Data Input Aplikasi Kredit
sample_instruction = "Anda adalah Senior Credit Risk Underwriter AI di institusi perbankan. Tugas Anda adalah mengevaluasi aplikasi kredit pemohon berdasarkan data demografi, keuangan, hasil prediksi model XGBoost (Probability of Default), dan kontribusi faktor risiko matematis (SHAP Values).\n\nHasilkan laporan analisis kredit (Credit Underwriting Memo) yang terstruktur strictly dalam format JSON valid."

sample_input = """### PROFIL PEMOHON PINJAMAN:
- Usia Pemohon: 24 tahun
- Pendapatan Tahunan: $45,000
- Status Kepemilikan Rumah: RENT
- Lama Bekerja: 2.0 tahun
- Tujuan Pinjaman: MEDICAL
- Peringkat Risiko Kredit (Grade): C
- Besaran Pinjaman yang Diajukan: $12,000
- Suku Bunga Pinjaman: 13.50%
- Rasio Pinjaman / Pendapatan: 26.7%
- Riwayat Gagal Bayar Sebelumnya: N
- Panjang Riwayat Kredit: 3 tahun

### KALKULASI RISIKO ML & ANALISIS SHAP:
- Prediksi Probability of Default (PD): 31.2%
- Kategori Risiko Awal: MEDIUM_RISK
- Rekomendasi Awal: MANUAL_REVIEW
- Faktor Pendorong Risiko Terbesar (+SHAP):
  * person_home_ownership_RENT bernilai 1.0 (SHAP: +0.28)
  * Rasio pinjaman terhadap pendapatan sebesar 26.7% (SHAP: +0.22)
- Faktor Pereda Risiko Terbesar (-SHAP):
  * Riwayat gagal bayar bersih (cb_person_default_on_file = N) (SHAP: -0.65)
  * Pendapatan tahunan sebesar $45,000 (SHAP: -0.35)"""

# 6.3 Susun Inference Prompt (Assistant kosong untuk digenerate)
inference_prompt = prompt_template.format(sample_instruction, sample_input, "").rstrip()

device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer([inference_prompt], return_tensors="pt").to(device)

print("Menjalankan inferensi model...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,                   # Suhu rendah untuk menjaga konsistensi format JSON
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# 6.4 Decode Output dan Ekstrak Respon Assistant
response_text = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
generated_response = response_text.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

print("\n=== HASIL RESPON MODEL ===")
print(generated_response)

# 6.5 Validasi Format JSON Output
print("\n=== VALIDASI FORMAT JSON ===")
try:
    parsed_json = json.loads(generated_response)
    print("[PASS] JSON VALID dan Terstruktur!")
    print(f"Rekomendasi Akhir : {parsed_json.get('recommendation')}")
    print(f"Tingkat Risiko    : {parsed_json.get('risk_level')}")
    print(f"Ringkasan         : {parsed_json.get('executive_summary')}")
except json.JSONDecodeError as e:
    print(f"[FAIL] Output bukan JSON valid: {e}")


### 7. Menyimpan LoRA Adapter & Export Merged Model
- **LoRA Adapter**: Bobot ringan (~30MB-50MB) yang dapat dimuat kapan saja bersama base model.
- **Merged Model (FP16)**: Menggabungkan bobot LoRA langsung ke dalam base model FP16 menjadi satu model utuh. Model ini dapat dikonversi ke format **GGUF** (untuk Ollama / llama.cpp) atau di-deploy langsung pada vLLM / Hugging Face Inference Endpoints.
> **Penting untuk GPU 4GB VRAM**: Proses merge disarankan dijalankan di RAM CPU (`device_map="cpu"`) untuk mencegah CUDA Out-of-Memory saat memuat unquantized model FP16.

In [ ]:
# ==============================================================================
# 7. SIMPAN LORA ADAPTER & MERGE KE BASE MODEL (FP16)
# ==============================================================================

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 7.1 Tentukan Direktori Penyimpanan Adapter
adapter_dir = os.path.abspath(os.path.join("..", "models", "lora_adapter")) if os.path.exists("../models") else os.path.abspath("models/lora_adapter")
os.makedirs(adapter_dir, exist_ok=True)

print(f"Menyimpan LoRA adapter ke: {adapter_dir}...")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"[SUCCESS] LoRA Adapter & Tokenizer berhasil disimpan di '{adapter_dir}'!")

# 7.2 Merge LoRA Adapter ke Base Model (Float16)
print("\n--- Memulai Proses Merge Model (Base Model + LoRA Adapter ke FP16) ---")

# Catatan: Merging model unquantized 3B butuh ~6GB RAM/VRAM.
# Di GPU 4GB VRAM, gunakan device_map="cpu" agar aman dari CUDA OOM.
merge_device = "cpu"  # Gunakan "cpu" untuk menghindari GPU OOM pada VRAM terbatas

print(f"Memuat base model unquantized (torch.float16) pada device: {merge_device}...")
base_model_reload = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map=merge_device,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

print("Memasang LoRA adapter dan melakukan merge_and_unload()...")
merged_model = PeftModel.from_pretrained(base_model_reload, adapter_dir)
merged_model = merged_model.merge_and_unload()

# 7.3 Simpan Merged Model
merged_dir = os.path.abspath(os.path.join("..", "models", "merged_model")) if os.path.exists("../models") else os.path.abspath("models/merged_model")
os.makedirs(merged_dir, exist_ok=True)

print(f"Menyimpan merged model ke: {merged_dir}...")
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

print(f"\n[SUCCESS] Merged Model berhasil disimpan di: '{merged_dir}'!")
print("Model siap dikonversi ke format GGUF atau di-deploy ke production.")
